# Hello Kaggle Community! 👋
# 🐈‍⬛ Black Cat SWE Agent | Second Strike
### Gemma 4 Developer Agent • scored anchor → testable challenger

> **Mission.** Turn a bug report into a small verified source patch. The evaluator applies each patch in a fresh repository and scores the fraction of issues whose tests pass.

**The first strike scored 0.06 public resolution rate.** This version is a challenger, not a claimed improvement. The scored agent is preserved as `anchor_submission.zip`; the new `submission.zip` is submitted for an independent leaderboard measurement. The public score uses roughly half of the hidden evaluation set, so its ordering can change on private data.

**Strategy at a glance**

| Stage | Agent action | Failure control |
|---|---|---|
| 🕵️ Decode | Strip PR-template boilerplate and identify the behavior in a terse issue | Do not give up because a hint is absent |
| 🔎 Locate | Search symbols, nearby tests, and relevant source with bounded calls | Fall back to source search if graph data is unavailable |
| 🧠 Diagnose | Trace the input to the first incorrect layer | Separate observed code from guesses |
| 🛠️ Repair | Make the smallest complete implementation change | Keep unrelated behavior stable |
| 🧪 Verify | Check the requested behavior and a boundary case when practical | Keep temporary scripts out of the patch |
| 📦 Submit | Review git diff and call `submit_patch()` explicitly | Preserve the scored anchor archive |

**What changed from the anchor.** The same harness-supported quantized Gemma 4 31B model and nine tools remain. The challenger uses a more explicit issue-decoding and verification policy, `HIGH` thinking with a 4,096-token budget, and a 16,384-token output ceiling. These changes are hypotheses, **not measured gains**. No private test data, external API, or LoRA weights are embedded here.

**What this notebook produces.** `/kaggle/working/submission.zip` contains only declarative YAML and a system prompt. The notebook checks it, displays public-data visuals, and creates scored-anchor and lightweight fallback archives. Kaggle evaluation, rather than this notebook, produces `submission.parquet` and a leaderboard score.

Sources: [competition overview](https://www.kaggle.com/competitions/gemma-4-developer-agent/overview), [data and harness guide](https://www.kaggle.com/competitions/gemma-4-developer-agent/data), [rules](https://www.kaggle.com/competitions/gemma-4-developer-agent/rules).

## 1 · Mission control

The diagram below maps the agent's route. It is not a performance chart. The challenger's resolution rate is unknown until the benchmark runs.

In [ ]:
from pathlib import Path
import json, zipfile, platform, textwrap
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

WORKING = Path('/kaggle/working') if platform.system() == 'Linux' and Path('/kaggle/working').exists() else Path.cwd()
WORKING.mkdir(parents=True, exist_ok=True)
print(r"""
 /\_/\\
( o.o )  BLACK CAT SWE AGENT // SECOND STRIKE
 > ^ <   issue detective -> bug hunter -> patch crafter -> test scout
         "No paws on the test files."  -- Chief Whisker Officer
""")
print(f'Python {platform.python_version()} | artifact directory: {WORKING}')

In [ ]:
stages = [
    ('01', 'DECODE', 'issue, minus boilerplate'),
    ('02', 'LOCATE', 'symbols + bounded search'),
    ('03', 'REPAIR', 'root-cause source patch'),
    ('04', 'VERIFY', 'behavior + boundary case'),
    ('05', 'SUBMIT', 'checked git diff'),
]
fig, ax = plt.subplots(figsize=(13, 2.8), facecolor='#0b1220')
ax.set_facecolor('#0b1220')
colors = ['#38bdf8', '#818cf8', '#a78bfa', '#34d399', '#fbbf24']
for i, (num, title, detail) in enumerate(stages):
    x = i * 2.5
    ax.scatter(x, 0, s=1100, color=colors[i], zorder=3, edgecolor='white', linewidth=1.2)
    ax.text(x, 0, num, ha='center', va='center', color='#0b1220', weight='bold', fontsize=12)
    ax.text(x, 0.85, title, ha='center', color='white', weight='bold', fontsize=11)
    ax.text(x, -0.85, detail, ha='center', color='#cbd5e1', fontsize=9)
    if i < len(stages) - 1:
        ax.annotate('', xy=(x+2.07, 0), xytext=(x+0.43, 0),
                    arrowprops=dict(arrowstyle='->', color='#64748b', lw=2))
ax.set_xlim(-0.9, 10.9)
ax.set_ylim(-1.4, 1.4)
ax.axis('off')
ax.set_title('One issue → one verified patch', color='white', fontsize=15, weight='bold', pad=15)
plt.show()

## 2 · Public development set

The public task index supports repository and issue-shape diagnostics. Reference patches are **never** packed into the agent archive. Missing data produces a clear skip message; archive generation remains runnable without it.

In [ ]:
candidates = [Path.cwd() / 'tasks.jsonl', Path('/kaggle/input/gemma-4-developer-agent/tasks.jsonl')]
input_root = Path('/kaggle/input')
if input_root.exists():
    candidates += list(input_root.rglob('tasks.jsonl'))
task_path = next((p for p in candidates if p.exists()), None)
tasks = None
if task_path:
    with task_path.open(encoding='utf-8') as handle:
        rows = [json.loads(line) for line in handle if line.strip()]
    tasks = pd.DataFrame({
        'id': [row['instance_id'] for row in rows],
        'repository': [row['repo'] for row in rows],
        'issue_chars': [len(row.get('problem_statement') or '') for row in rows],
        'reference_patch_bytes': [len((row.get('patch') or '').encode('utf-8')) for row in rows],
    })
    assert tasks['id'].is_unique
    print(f'Public tasks: {len(tasks)} | repositories: {tasks.repository.nunique()} | source: {task_path.name}')
    display(tasks.groupby('repository').agg(tasks=('id','size'),
        median_issue_chars=('issue_chars','median'),
        median_reference_patch_bytes=('reference_patch_bytes','median')).sort_values('tasks', ascending=False))
else:
    print('Public tasks.jsonl was not attached. Add the competition data to enable the EDA figures.')

In [ ]:
if tasks is not None:
    counts = tasks['repository'].value_counts().sort_values()
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.4), facecolor='#0b1220')
    for ax in axes:
        ax.set_facecolor('#111c2e')
        ax.tick_params(colors='#cbd5e1', labelsize=9)
        for spine in ax.spines.values(): spine.set_visible(False)
    axes[0].barh(counts.index, counts.values, color='#38bdf8')
    for i, value in enumerate(counts.values):
        axes[0].text(value + 0.6, i, str(value), color='white', va='center')
    axes[0].set_xlim(0, counts.max() * 1.18)
    axes[0].set_title('Public development issues by repository', color='white', weight='bold')
    axes[0].set_xlabel('Issue count', color='#cbd5e1')
    groups = [tasks.loc[tasks.repository == repo, 'issue_chars'].values for repo in counts.index]
    box = axes[1].boxplot(groups, vert=False, tick_labels=counts.index, patch_artist=True,
                          showfliers=False, medianprops=dict(color='#fbbf24', lw=2))
    for patch in box['boxes']: patch.set_facecolor('#818cf8')
    for item in box['whiskers'] + box['caps']: item.set_color('#cbd5e1')
    axes[1].set_title('Problem statement length by repository', color='white', weight='bold')
    axes[1].set_xlabel('Characters; boxes show IQR, outliers hidden', color='#cbd5e1')
    fig.suptitle('Public data describes development coverage, not hidden-test performance',
                 color='white', fontsize=14, weight='bold', y=1.03)
    fig.tight_layout()
    plt.show()

## 3 · Anchor → challenger

The chart records the **observed** public score for the prior submission. The challenger has no score yet; no projected bar is drawn. A higher public score is evidence for a promising challenger, while the private split remains uncertain.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 2.8), facecolor='#0b1220')
ax.set_facecolor('#111c2e')
ax.barh(['Scored anchor'], [0.06], height=0.38, color='#38bdf8')
ax.text(0.061, 0, '0.06', va='center', color='white', weight='bold')
ax.set_xlim(0, 0.15)
ax.set_xlabel('Public resolution rate', color='#cbd5e1')
ax.set_title('Verified anchor; challenger pending evaluation', color='white', weight='bold')
ax.tick_params(colors='#cbd5e1')
for spine in ax.spines.values(): spine.set_visible(False)
fig.tight_layout()
plt.show()

## 4 · Build the agent

Only host-registered tools appear in the YAML. The archive uses one declared Gemma 4 model and no unvalidated adapter weights. The prompt adapts to unseen private repositories rather than memorizing public patches.

In [ ]:
AGENT_YAML = 'name: black_cat_swe_agent_v2\nmodel: gemma-4-31b-it-qat-w4a16-ct\ndescription: Issue-driven software engineer with focused verification and explicit patch submission.\ninstruction: !include prompts/system.md\ntools:\n  - run_command\n  - read_file\n  - edit_file\n  - write_file\n  - get_status\n  - submit_patch\n  - get_code_neighbors\n  - search_similar_code\n  - get_code_subgraph\ngenerate_content_config: !include configs/sampling.yaml\n'
SAMPLING_YAML = 'temperature: 0.10\ntop_p: 0.90\nmax_output_tokens: 16384\nseed: 42\nthinking_config:\n  thinking_level: HIGH\n  thinking_budget: 4096\n  include_thoughts: false\n'
SYSTEM_PROMPT = 'You are Black Cat, an autonomous software engineer solving one repository issue in /workspace. Your output is the actual git patch, not a proposed plan. The evaluator applies that patch to a fresh checkout and runs separate tests. Finish by calling submit_patch() after inspecting the diff.\n\nMISSION: resolve the behavior requested by the issue with the smallest complete implementation. Spend the available budget on evidence and a working patch. Never claim a test passed unless a tool showed that result.\n\n1. Decode the issue. Extract the requested behavior, affected API, edge cases, and any literal error text. Some reports are copied from pull-request templates: ignore boilerplate about discussions, checklists, and release process; use the substantive title and description. A terse title is still an actionable specification. If it names a method, exception, or symptom, find the implementation and infer a concrete failing case from current code. Do not stop merely because no test or hint was supplied.\n\n2. Locate efficiently. The initial user message includes a repository listing. Start with the named module or symbol if present. Otherwise use one bounded `rg -n` search for exact terms, then search nearby API names or tests if the first route fails. If `rg` is unavailable, use bounded `grep -R`. Read focused file ranges, not entire large files. Use code-graph tools only when the graph exists and a real symbol name is known. `search_similar_code` resolves symbols, not natural-language queries. If a graph call errors, switch immediately to source search. Search both implementation and the nearest relevant tests or examples, especially public behavior and existing conventions.\n\n3. Form a testable hypothesis. Trace inputs through the affected function and its callers until you can explain why the current code fails the requested behavior. Prefer fixing the earliest correct layer rather than patching a downstream symptom. For compatibility changes, examine both the ordinary path and boundary cases: empty input, missing fields, type variations, encoding, and exception behavior as relevant. Keep public signatures and existing behavior stable outside the issue.\n\n4. Implement a complete, narrow fix. Use `edit_file` for focused replacements; if matching fails or is ambiguous, reread a smaller range and retry. Use `write_file` only when a new implementation file is genuinely needed. It is valid to change runtime Python, Python examples such as `docs_src`, or package metadata when the requested behavior requires it. Do not modify tests to satisfy them. Never edit harness-generated `pytest.ini` or `conftest.py`, hidden tests, reference patches, or future git history. Avoid broad reformatting and unrelated cleanup.\n\n5. Verify the actual behavior. First run a small reproduction or a targeted existing test tied to the issue. Prefer `python3 -c` for a tiny isolated check; use `python3 -m pytest path/to/test.py::test_name -q` when a relevant test exists. Do not run the full repository suite, install dependencies, or use the network. If a check fails, read the failure and repair a regression or incorrect assumption. Distinguish unrelated environment errors from behavior failures. For a broad feature, verify at least one positive and one boundary case when practical. Keep temporary reproduction scripts in /tmp so they cannot enter the patch.\n\n6. Review before submission. Run `git diff --check`, `git status --short`, and `git diff HEAD` (narrow the diff display if large). Ensure every changed file is intentional and that the patch contains implementation, not scratch files. Call `get_status()` when budget is uncertain. Reserve enough time for this review and the final tool action. If time is tight, submit the best defensible working edit; do not abandon a plausible fix solely because optional tests cannot run.\n\n7. Call `submit_patch()` exactly once as the final tool action. It is free and captures git changes, including intended new files. Check its returned patch size and file count. Then give a concise final text summary of what changed and what was verified.\n\nTool behavior: commands have a 300-second ceiling and truncated output; file reads are capped. Keep commands and edits small. The environment is offline and dependencies are already staged. Use tools promptly rather than spending long turns on speculative prose. If a tool call fails, inspect the error and take a simpler route. Do not fabricate source content or test outcomes.\n'
ANCHOR_YAML = 'name: black_cat_swe_agent\nmodel: gemma-4-31b-it-qat-w4a16-ct\ndescription: Repository-agnostic software engineering agent for focused Python fixes.\ninstruction: !include prompts/system.md\ntools:\n  - run_command\n  - read_file\n  - edit_file\n  - write_file\n  - get_status\n  - submit_patch\n  - get_code_neighbors\n  - search_similar_code\n  - get_code_subgraph\ngenerate_content_config: !include configs/sampling.yaml\n'
ANCHOR_SAMPLING = 'temperature: 0.15\ntop_p: 0.90\nmax_output_tokens: 8192\nseed: 42\nthinking_config:\n  thinking_level: MEDIUM\n  thinking_budget: 2048\n  include_thoughts: false\n'
ANCHOR_PROMPT = 'You are Black Cat, an autonomous software engineer working on one Python repository issue. Your deliverable is a minimal, working source-code patch under /workspace. The benchmark checks the patch in a fresh checkout with separate tests. Work independently and finish by calling submit_patch().\n\nOperating priorities: correctness, focused navigation, reproducible edits, and useful work within the time and tool budget. Never invent a test result or claim a file was changed before a tool confirms it.\n\n1. Orient quickly. Read the problem statement and hints. Identify exact error strings, public behavior, expected types, and likely symbols. The initial prompt includes a workspace listing. If it identifies an obvious file, read that file. Otherwise make one bounded discovery call using run_command with rg over relevant Python paths. Use search_similar_code only with a real symbol name, not a natural-language sentence; it resolves existing graph nodes, not arbitrary queries. Use graph neighbors when a caller or dependency relationship changes the fix. If graph tools are missing or return errors, use rg and read_file.\n2. Diagnose before editing. Inspect the implementation and the nearest relevant tests or examples. State a short hypothesis internally and distinguish observed behavior from guesses. Favor one small source edit over broad rewrites. Preserve signatures, existing conventions, compatibility, and error wording required by the issue. For feature requests, implement the specified behavior end to end, including edge cases that the issue names.\n3. Edit precisely with edit_file. If a matching block is ambiguous, read a narrower range before retrying. Use write_file only for a new implementation file or when a narrow replacement is impossible. Do not alter tests, snapshots, generated files, pytest.ini, conftest.py, package metadata, or unrelated formatting to make checks pass. In tasks involving executable documentation examples, docs_src or equivalent example source may be the implementation target.\n4. Verify economically. Prefer a targeted existing test or a tiny inline reproduction for the changed behavior. Specify the exact test module or test name; never run an entire repository test suite. Commands have a 300-second ceiling and truncated output. Do not install packages or access the network. If a test fails, diagnose whether it is related to the change. Fix a real regression if time permits. Ignore unrelated pre-existing environment failures after recording the uncertainty internally.\n5. Inspect git diff --check and git diff --stat, then inspect the relevant diff. Remove temporary files from /workspace; put scratch scripts in /tmp. Do not put expected-answer strings or issue-specific hardcoded patches into the repository. Never inspect hidden solution files or future commits. If time or tool calls run low, preserve the best working edit and finish promptly.\n6. Call submit_patch() as the final tool action. It captures git diff HEAD, including intended new files. Check the returned patch_size and files_changed. If no defensible edit is possible, submit the empty state honestly rather than inventing a change. Give a concise final summary with the actual verification status.\n\nTool budget guide: spend roughly 2-4 calls locating code, 2-5 calls implementing, and 1-3 calls validating. This is a guide, not a hard limit; complex issues may need more. Do not use an analysis sub-agent unless one is provided by the configuration. The harness may continue a turn if you stop without submit_patch(), so make that call explicitly.\n'


In [ ]:
PRIMARY_FILES = {
    'agent.yaml': AGENT_YAML,
    'configs/sampling.yaml': SAMPLING_YAML,
    'prompts/system.md': SYSTEM_PROMPT,
}
ANCHOR_FILES = {
    'agent.yaml': ANCHOR_YAML,
    'configs/sampling.yaml': ANCHOR_SAMPLING,
    'prompts/system.md': ANCHOR_PROMPT,
}
FALLBACK_FILES = {
    'agent.yaml': 'name: black_cat_fallback\nmodel: gemma-4-31b-it-qat-w4a16-ct\ninstruction: !include prompts/system.md\ntools:\n  - run_command\n  - read_file\n  - edit_file\n  - get_status\n  - submit_patch\ngenerate_content_config:\n  temperature: 0.1\n  max_output_tokens: 4096\n',
    'prompts/system.md': 'Solve the Python repository issue under /workspace. Inspect the relevant source, make the smallest correct source edit, run a focused check when possible, inspect git diff, and call submit_patch(). Do not alter tests or harness files. Work offline, avoid full-suite tests, and report only verified facts.\n',
}

def validate_files(files):
    assert set(files) and 'agent.yaml' in files
    allowed = {'.yaml', '.yml', '.md', '.txt', '.py', '.json', '.safetensors'}
    for name, content in files.items():
        path = Path(name)
        assert not path.is_absolute() and '..' not in path.parts
        assert path.suffix in allowed and isinstance(content, str) and content.strip()
    root_text = files['agent.yaml']
    for line in root_text.splitlines():
        if '!include' in line:
            included = line.split('!include', 1)[1].strip()
            assert included in files, f'Missing include: {included}'
    parsed = yaml.safe_load(root_text.replace('!include ', ''))
    assert parsed['model'] == 'gemma-4-31b-it-qat-w4a16-ct'
    assert 'submit_patch' in parsed['tools']
    assert len(files) <= 10000
    return parsed

def write_archive(files, output):
    validate_files(files)
    with zipfile.ZipFile(output, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        for name, content in sorted(files.items()):
            archive.writestr(name, content.encode('utf-8'))
    with zipfile.ZipFile(output) as archive:
        assert archive.testzip() is None
        assert sorted(archive.namelist()) == sorted(files)
        assert archive.namelist().count('agent.yaml') == 1
    return output

primary = write_archive(PRIMARY_FILES, WORKING / 'submission.zip')
anchor = write_archive(ANCHOR_FILES, WORKING / 'anchor_submission.zip')
fallback = write_archive(FALLBACK_FILES, WORKING / 'fallback_submission.zip')
print(f'Primary archive: {primary} ({primary.stat().st_size:,} bytes)')
print(f'Scored anchor archive: {anchor} ({anchor.stat().st_size:,} bytes)')
print(f'Fallback archive: {fallback} ({fallback.stat().st_size:,} bytes)')

## 5 · Release gate

This checks structure and packaging. A resolved-issue score requires the separate evaluator, which is unavailable in this notebook. The scored anchor and a minimal fallback remain available if the challenger fails to load or scores worse.

In [ ]:
with zipfile.ZipFile(primary) as archive:
    names = archive.namelist()
    print('Archive members:', ', '.join(names))
    assert names == sorted(PRIMARY_FILES)
    assert not any(name.startswith('adapters/') for name in names)
    assert not any('tasks.jsonl' in name or 'patch' in name for name in names)
print(r"""
 /\_/\\
( o.o )  PACKAGE CHECK PASSED
 > ^ <   challenger submission.zip is ready for the evaluator
         "I inspected the diff. The diff inspected me back."
         Challenger resolution rate: pending Kaggle scoring
""")

## 6 · Handoff

**Selected strategy for this trial:** quantized Gemma 4 31B, one focused agent, issue decoding, root-cause search, targeted verification, and explicit patch submission. The archive is statically checked and rerunnable from this notebook. **Score uncertainty:** the anchor scored 0.06 public; the challenger has no measured resolution score until the next submission completes. The scored anchor stays available as `anchor_submission.zip`, with a minimal packaging fallback as a third option.

The notebook is published with the competition dataset attached. Public task summaries are shown; full reference patches and test patches are never printed or added to `submission.zip`.